In [1]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv()

d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm_model = init_chat_model(model="openai/gpt-oss-120b", model_provider="groq")

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "sample_kb/kb",
    glob= "**/*.md",
    loader_cls=TextLoader,
    recursive=True
)

docs = loader.load()

In [4]:
from pathlib import Path
from datetime import datetime

for doc in docs:
    path = Path(doc.metadata["source"])

    updated = datetime.fromtimestamp(path.stat().st_mtime)

    parts = path.parts

    tenant = parts[2]

    doc.metadata["updated_at"] = updated.isoformat()
    doc.metadata["tenant"] = tenant

In [5]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

header_to_split = [("#","h1"),("##","h2"),("###","h3")]
md_splitter = MarkdownHeaderTextSplitter(header_to_split)

header_chunks = []

for doc in docs:
    chunks = md_splitter.split_text(doc.page_content)

    for chunk in chunks:
        chunk.metadata.update(doc.metadata)
    
    header_chunks.extend(chunks)


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

embeddings = HuggingFaceEmbeddings(model_name = "BAAI/bge-large-en-v1.5", encode_kwargs= {"normalize_embeddings": True})

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

semantic_chunks = []

for chunk in header_chunks:
    docs = semantic_splitter.create_documents(
        [chunk.page_content],
        metadatas=[chunk.metadata]
    )
    total = len(docs)
    for i, doc in enumerate(docs):
        doc.metadata["chunk_index"] = i
        doc.metadata["total_chunks"] = total
    semantic_chunks.extend(docs)

#convert tables to readable text
for chunk in semantic_chunks:
    if "|" in chunk.page_content and "---" in chunk.page_content:
        lines = chunk.page_content.strip().split("\n")
        table_lines = [l for l in lines if l.strip().startswith("|")]
        if len(table_lines) >= 3:
            headers = [h.strip() for h in table_lines[0].split("|") if h.strip()]
            rows = []
            for row in table_lines[2:]:
                cells = [c.strip() for c in row.split("|") if c.strip()]
                if len(cells) == len(headers):
                    rows.append(". ".join(f"{headers[j]}: {cells[j]}" for j in range(len(headers))) + ".")
            non_table = [l for l in lines if not l.strip().startswith("|")]
            chunk.page_content = "\n".join(rows) + ("\n\n" + "\n".join(non_table) if non_table else "")


C:\Users\athar\AppData\Local\Temp\ipykernel_20472\1427635226.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4226.12it/s]


In [7]:
from qdrant_client import QdrantClient

client = QdrantClient(
    url="https://aff5a553-d08d-4b6f-a735-a9bbabf90785.eu-central-1-0.aws.cloud.qdrant.io",
    api_key=os.getenv("QDRANT_API_KEY")
)

from langchain_qdrant import QdrantVectorStore

vector_store = QdrantVectorStore(
    client=client,
    collection_name="knowledge_base",
    embedding=embeddings,
    vector_name="vectors"
)

In [8]:
from langchain_classic.indexes import SQLRecordManager, index

record_manager = SQLRecordManager(
    namespace = "qdrant/knowledge_base",
    db_url = os.getenv("POSTGRES_DB_URL")
)

record_manager.create_schema()

index(
    docs_source=semantic_chunks,
    record_manager=record_manager,
    vector_store=vector_store,
    cleanup="incremental",
    source_id_key="source"
)

d:\RAG\.venv\Lib\site-packages\langchain_core\indexing\api.py:409: UserWarning: Using SHA-1 for document hashing. SHA-1 is *not* collision-resistant; a motivated attacker can construct distinct inputs that map to the same fingerprint. If this matters in your threat model, switch to a stronger algorithm such as 'blake2b', 'sha256', or 'sha512' by specifying  `key_encoder` parameter in the `index` or `aindex` function. 
  _warn_about_sha1()


{'num_added': 1, 'num_updated': 0, 'num_skipped': 136, 'num_deleted': 1}

In [9]:
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings

store = LocalFileStore("./embedding_cache/")
cache_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, store, namespace="BAAI/bge-large-en-v1.5"
)

d:\RAG\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [ ]:
logged_in_user = {
    "name": "atharva",
    "tenant": "engineering"
}

tenant = logged_in_user["tenant"]

tenant_chunks = [docs for docs in semantic_chunks if docs.metadata["tenant"] == tenant]

In [11]:
from qdrant_client.models import Filter, FieldCondition, MatchAny
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever


sparse_retriever = BM25Retriever.from_documents(semantic_chunks)
sparse_retriever.k = 5

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k":15,})
        # "filter": Filter(
        #     must=[
        #         FieldCondition(
        #             key="metadata.tenant",
        #             match=MatchAny(any=[tenant])
        #         )
        #     ]
        # )
        # })


hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.6,0.4]
)

from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever

model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=model, top_n=5)
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=hybrid_retriever)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4728.35it/s]


In [12]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise assistant that answers ONLY using the provided context.\n"
     "Rules:\n"
     "1. If the context does not contain the answer, say so explicitly — never guess.\n"
     "2. Every factual claim must be traceable to a numbered source below.\n"
     "3. Cite sources inline like [1], [2] matching the numbering given.\n"
     "4. Be concise. Do not repeat the question.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

In [13]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs):
    return "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))

rag_chain = (
    {"context": compression_retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm_model
    | StrOutputParser()
)


In [14]:
eval_questions = [
    "What is the refund window for a Growth plan customer?",
    "How many business days does refund processing take?",
    "If I cancel my annual plan early, is there any fee besides the prorated refund?",
    "What's the difference between ERR-5310 and ERR-5311?",
    "If a customer sees ERR-9001, what should support tell them?",
    "A Growth plan customer's uptime was 99.6% this month against a 99.9% commitment. What does the SLA rule say about credits per shortfall?",
    "A Starter plan customer's uptime dropped to 98%. What service credit do they get?",
    "What happens to webhook events if a customer's endpoint goes down?",
    "Has a webhook delivery outage actually happened before, and how was it resolved?",
    "What caused the March 2026 checkout outage?",
    "What changed after the March 2026 checkout outage to prevent a repeat?",
    "What triggers a SEV-1 for payment-related incidents?",
    "As a security team member, what's tracked under JIRA-SEC-4471?",
]

eval_ground_truths = [
    "30 days from the charge.",
    "5-7 business days, reduced from the prior 10-business-day policy after a payment-processor upgrade.",
    "Yes, a 5% early-termination administrative fee, introduced in the current (v2) policy. The prior policy had no such fee.",
    "ERR-5310 (risk_declined) means the risk engine declined the transaction outright. ERR-5311 (risk_manual_review) means it was flagged for manual review, not declined.",
    "Customers should never see ERR-9001 directly. It is translated to a generic ERR-5001 message externally, while the real code (internal_config_error) is logged internally for engineering.",
    "The SLA gives 10% credit per 0.1% shortfall below the uptime commitment.",
    "None. The Starter plan has no uptime commitment and no service credit provision.",
    "Events are retried on an exponential backoff schedule for up to 3 days and are not dropped.",
    "Yes — ticket #51087: a customer's TLS certificate rotation broke delivery for about 6 hours; 14 queued events were manually redelivered once the cert chain was fixed.",
    "A regional outage at the primary card-network processor caused ERR-4092 timeouts.",
    "The circuit-breaker trip threshold was reduced from 15% to 8% error rate (shipped 2026-03-22), and automated failover to replace the manual flag was planned for Q3 2026.",
    "A SEV-1 is triggered when the ERR-4092 rate exceeds 10% of total payment volume for more than 5 minutes and failover does not resolve it.",
    "An open control exception: automated de-provisioning of contractor accounts isn't fully automated yet. Remediation is targeted for the Q3 2026 SOC 2 audit cycle.",
]

from ragas import EvaluationDataset

eval_data = []

for q,gt in zip(eval_questions, eval_ground_truths):
    retrieved_docs = compression_retriever.invoke(q)
    contexts = [d.page_content for d in retrieved_docs]
    answer = rag_chain.invoke(q)

    eval_data.append({
        "user_input": q,
        "response": answer,
        "retrieved_contexts": contexts,
        "reference": gt,
    })

    
eval_dataset = EvaluationDataset.from_list(eval_data)

In [19]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_model = init_chat_model("llama-3.1-8b-instant", model_provider="groq")

ragas_llm = LangchainLLMWrapper(ragas_model)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

C:\Users\athar\AppData\Local\Temp\ipykernel_20472\4064373644.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ragas_model)
C:\Users\athar\AppData\Local\Temp\ipykernel_20472\4064373644.py:7: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [20]:
from ragas import evaluate
from ragas.run_config import RunConfig

from ragas.metrics import _faithfulness, _answer_relevancy, _context_precision, _context_recall

_answer_relevancy.strictness = 1

ragas_results = evaluate(
    dataset=eval_dataset,
    metrics=[_faithfulness,_answer_relevancy,_context_precision, _context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=RunConfig(max_workers=1, timeout=200),
)

print(ragas_results)
df = ragas_results.to_pandas()
df.to_csv("ragas_eval_results.csv", index=False)

Evaluating: 100%|██████████| 52/52 [34:53<00:00, 40.25s/it]

{'faithfulness': 0.9038, 'answer_relevancy': 0.8506, 'context_precision': 0.8449, 'context_recall': 0.7487}
